In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"I:\Use Case\Retails-Sales-Analytics\data\raw\sample_-_superstore.xls")

In [4]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


`Column Standardization`

Objective:-

*Standardize all column names into snake_case to improve readability and maintain consistency across Python, SQL, and Power BI workflows.

Cleaning Performed:-

* Converted all column names to lowercase.
* Replaced spaces with underscores (_).
* Replaced hyphens (-) with underscores.
* Replaced forward slashes (/) with underscores.

Outcome

All column names now follow Python naming conventions.
The dataset is ready for feature engineering and downstream analysis.

In [5]:
df= df.rename(columns=lambda x: x.strip()
                                .lower()
                                .replace(" ", "_")
                                .replace("-", "_")
                                .replace("/", "_" ),
                                )
df.to_excel("../data/cleaned/sample_-_superstore.xlsx", index=False)

In [7]:
df.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country_region',
 'city',
 'state_province',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

VERIFY THE SAVED FILE

In [9]:
import os

os.path.exists("../data/cleaned/sample_-_superstore.xlsx")

True

Explicit conversion of date columns into DateTime

In [10]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

`Feature Engineering`

Objective:-

    Create new features that improve business analysis and make downstream SQL queries and Power BI dashboards more meaningful.

Features Created:-

    order_year
    order_month
    order_day
    shipping_days
    profit_margin

Business Value

    Enables time-based trend analysis.
    Measures shipping efficiency.
    Improves profitability analysis across products and regions.
    Prepares the dataset for dashboarding and business reporting.

In [14]:
df["order_year"] = df["order_date"].dt.year

In [ ]:
df["order_month"] = df["order_date"].dt.month_name()


In [33]:
df["order_month_num"] = df["order_date"].dt.month

In [15]:
df["order_day"] = df["order_date"].dt.day_name()

In [16]:
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days

In [17]:
df["profit_margin"] = (df["profit"] / df["sales"])

In [25]:
df["is_loss"] = np.where(df["profit"] < 0, "Yes", "No")

HANDLE DIVISION BY ZERO

In [18]:
df["profit_margin"] = np.where(
    df["sales"] != 0,
    df["profit"] / df["sales"],
    np.nan
)

VERIFY THE NEW COLUMNS

In [34]:
df[["order_day",
    "shipping_days",
    "profit_margin",
    "order_year",
    "order_month",
    "ship_date",
    "order_date",
    "sales",
    "profit",
    "order_month_num",
    "is_loss"]].head()

,order_day,shipping_days,profit_margin,order_year,order_month,ship_date,order_date,sales,profit,order_month_num,is_loss
0,Tuesday,4,0.3375,2023,January,2023-01-07,2023-01-03,16.448,5.5512,1,No
1,Wednesday,4,-1.5500,2023,January,2023-01-08,2023-01-04,3.540,-5.4870,1,Yes
2,Wednesday,4,0.3625,2023,January,2023-01-08,2023-01-04,11.784,4.2717,1,No
3,Wednesday,4,-0.2375,2023,January,2023-01-08,2023-01-04,272.736,-64.7748,1,Yes
4,Thursday,7,0.2500,2023,January,2023-01-12,2023-01-05,19.536,4.8840,1,No


SAVE THE UPDATED DATASET

In [ ]:
df.to_excel(
    "../data/cleaned/sample_-_superstore.xlsx",
    index=False
)